# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a dataset using the `mlcroissant` library, while referencing Croissant entities by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. The dataset metadata contains fields such as `@id`, `name`, and `description`. Here we load the metadata and print the dataset's title and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their `@id`s, and column IDs. `mlcroissant` exposes all Croissant entities, so you can inspect them using their `@id`.
Below, we enumerate available record sets (`cr:RecordSet`), their fields, and columns, with all references using the `@id` attribute.

In [ ]:
# Iterate through record sets using their @id
record_sets = list(dataset.record_sets)
print("Available Record Sets (referenced by @id):")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', '<no name>')}")
    # List fields
    if 'field' in rs:
        print("  Fields:")
        for field in rs['field']:
            print(f"    - @id: {field['@id']}, name: {field.get('name', '<no name>')}, dataType: {field.get('dataType', '<no dataType>')}")
    # List columns
    if 'column' in rs:
        print("  Columns:")
        for col in rs['column']:
            print(f"    - @id: {col['@id']}, name: {col.get('name', '<no name>')}")
print("\n---\n")

# For demonstration, print sample records from the first record set
if len(record_sets) > 0:
    first_rs_id = record_sets[0]['@id']
    print(f"Sample records from record_set @id={first_rs_id}:")
    for rec in dataset.records(record_set=first_rs_id):
        print(rec)
        break  # only show first record for brevity

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s obtained above. All entity references below use their `@id`.

In [ ]:
# Prepare to extract all record sets
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Show columns for first record set
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if main_record_set_id:
    print(f"Columns for record_set @id={main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    print("\nSample records:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on numeric criteria, normalizing numeric fields, and grouping records. All references are made using the Croissant `@id` attributes.

Below, we select a numeric field such as the GAD-7 score (commonly used in mental health datasets), filter records, normalize values, and group the filtered records by a demographic field (e.g., gender or education level).

In [ ]:
# Select a numeric field for analysis
# Find a GAD-7 score field by @id
numeric_field_id = None
group_field_id = None
if main_record_set_id:
    # Attempt to auto-detect likely numeric and group fields
    cols = dataframes[main_record_set_id].columns
    # Pick GAD-7 score (contains 'gad_7' or 'GAD_7')
    for col in cols:
        if 'gad_7' in col.lower() or 'GAD_7' in col:
            numeric_field_id = col
            break
    # Pick gender field (contains 'gender') or education ('education')
    for col in cols:
        if 'gender' in col.lower():
            group_field_id = col
            break
        if 'education' in col.lower():
            group_field_id = col
            break
    # Show available selections
    print(f"Numeric field @id: {numeric_field_id}")
    print(f"Group field @id: {group_field_id}")

    # Filtering records based on a threshold
    if numeric_field_id:
        threshold = 10
        df = dataframes[main_record_set_id]
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by demographic field
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields, such as the distribution of GAD-7 scores or comparisons across demographic groups. Below, we generate visualizations using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of GAD-7 scores
if main_record_set_id and numeric_field_id:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (GAD-7 scores)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot of GAD-7 scores by group
    if group_field_id:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we:
- Demonstrated loading a Croissant dataset using its schema URL and exploring metadata.
- Enumerated record sets, fields, and columns using their `@id` attributes, supporting FAIR principles.
- Loaded survey records and performed basic filtering, normalization, and grouping using key fields referenced by `@id`.
- Visualized numeric scores and their distributions, showing how demographic factors might relate to mental health indicators.

This workflow provides a FAIR-compliant, AI-ready pipeline for processing survey datasets using the `mlcroissant` library, referencing all entities by their unique Croissant `@id`.
